# U23 — Cloud Storage & Databases (Part 1): Lab

### Real-world brief: a storage layer for an engineering asset & inspection platform

A maintenance team has an **asset register** (pumps, motors, valves…) and a long **inspection history**. They need to store reports as files, query records transactionally, model the data two ways (relational vs document), and understand how it would **shard** across machines as it grows.

You'll work hands-on with the storage ideas from U23 using only the Python standard library (`sqlite3`, filesystem, JSON) — no cloud account required, but every concept maps directly to S3, RDS, DynamoDB and friends:
- **Object storage** — a local bucket with keys, metadata and tiering
- **Relational (SQLite)** — schema, ACID transactions, indexing
- **NoSQL document modelling** — the same data as JSON documents
- **Sharding & replication** — routing keys across shards; replica lag

**Resources provided:** `assets.csv` (200 assets) and `inspections.csv` (~4.9k inspections).

_Phase F — Data Engineering._

#objectives

Build an object store with keys, metadata and storage tiers

Model relational tables and run ACID transactions (commit/rollback)

Measure how an index speeds queries; read a query plan

Model the same data as a NoSQL document and compare trade-offs

Shard data by key across nodes and reason about skew & replication

#how to use this lab

Worked demos teach the pattern; 🧪 LAB EXERCISE cells are real tasks — replace `# YOUR CODE HERE`. Run top to bottom with Shift+Enter.

In [ ]:
# === SETUP: build the source files if missing ===
import os
import numpy as np
import pandas as pd


def build_assets(assets_path="assets.csv", insp_path="inspections.csv", seed=223, verbose=False):
    """Engineering asset register + inspection history for the storage/database lab (U23).

      assets.csv       one row per physical asset (master data)
      inspections.csv  many inspection events per asset (transactional data)

    Two related tables — perfect for relational modelling (joins, indexing, transactions),
    document modelling (asset + its inspections as one JSON doc) and sharding by asset key.
    """
    rng = np.random.default_rng(seed)

    NA = 200
    types = rng.choice(["pump", "valve", "motor", "compressor", "heat_exchanger"], NA,
                        p=[0.3, 0.25, 0.2, 0.15, 0.1])
    sites = rng.choice(["Plant_A", "Plant_B", "Plant_C"], NA, p=[0.45, 0.35, 0.2])
    assets = pd.DataFrame({
        "asset_id": [f"A{i:04d}" for i in range(1, NA + 1)],
        "asset_type": types, "site": sites,
        "install_date": pd.to_datetime("2015-01-01") + pd.to_timedelta(rng.integers(0, 3300, NA), unit="D"),
        "criticality": rng.choice(["low", "medium", "high"], NA, p=[0.4, 0.4, 0.2]),
        "manufacturer": rng.choice(["Acme", "Bharat", "Crompton", "Danfoss"], NA),
    })

    # inspections: variable count per asset
    rows = []
    iid = 1
    inspectors = ["E. Rao", "S. Khan", "M. Iyer", "P. Singh", "L. Dsouza"]
    for _, a in assets.iterrows():
        k = rng.integers(5, 45)                       # number of inspections
        dates = pd.to_datetime("2022-01-01") + pd.to_timedelta(np.sort(rng.integers(0, 850, k)), unit="D")
        crit_w = {"low": 0.05, "medium": 0.12, "high": 0.22}[a.criticality]
        for d in dates:
            fail = rng.random() < crit_w
            result = "fail" if fail else ("repair" if rng.random() < 0.15 else "pass")
            rows.append({
                "inspection_id": f"I{iid:06d}", "asset_id": a.asset_id, "inspection_date": d,
                "inspector": rng.choice(inspectors),
                "result": result,
                "health_score": round(float(np.clip(rng.normal(85 - (25 if fail else 0), 8), 0, 100)), 1),
                "duration_min": int(np.clip(rng.normal(45, 15), 5, None)),
                "cost_inr": int(np.clip(rng.gamma(2.0, 1800), 200, None)),
            })
            iid += 1
    insp = pd.DataFrame(rows)

    assets.to_csv(assets_path, index=False)
    insp.to_csv(insp_path, index=False)
    if verbose:
        print("assets:", assets.shape, "| inspections:", insp.shape)
        print("inspections per asset: min/median/max =",
              insp.groupby("asset_id").size().agg(["min", "median", "max"]).to_dict())
        print("result mix:", insp.result.value_counts().to_dict())
        print("site distribution:", assets.site.value_counts().to_dict())
    return assets, insp

if not (os.path.exists('assets.csv') and os.path.exists('inspections.csv')):
    build_assets(); print('Generated source files.')
else:
    print('Found the provided source files.')

In [ ]:
import pandas as pd, numpy as np, sqlite3, json, time, os, shutil
assets = pd.read_csv('assets.csv', parse_dates=['install_date'])
insp = pd.read_csv('inspections.csv', parse_dates=['inspection_date'])
print('assets:', assets.shape, '| inspections:', insp.shape)
assets.head(3)

# Part A — Object storage
Object storage (think S3) keeps each file as an **object** — bytes plus metadata — under a unique **key**, in a flat namespace called a **bucket**. We'll simulate one with a directory.

In [ ]:
# -----------------------------------------------------------
# 🔹 A1. A LOCAL OBJECT STORE: put / get / list / tier
# -----------------------------------------------------------
class ObjectStore:
    def __init__(self, bucket='bucket'):
        self.root = bucket; shutil.rmtree(bucket, ignore_errors=True); os.makedirs(bucket)
        self.meta = {}        # key -> metadata dict (tier, size, content_type)
    def put(self, key, data, tier='hot', content_type='application/json'):
        path = os.path.join(self.root, key.replace('/', '__'))
        blob = json.dumps(data).encode()
        with open(path, 'wb') as f: f.write(blob)
        self.meta[key] = {'tier': tier, 'size': len(blob), 'content_type': content_type}
    def get(self, key):
        path = os.path.join(self.root, key.replace('/', '__'))
        with open(path, 'rb') as f: return json.loads(f.read())
    def list(self, prefix=''):
        return sorted(k for k in self.meta if k.startswith(prefix))
    def set_tier(self, key, tier):
        self.meta[key]['tier'] = tier

store = ObjectStore('inspection_bucket')
print('empty bucket created.')

In [ ]:
# -----------------------------------------------------------
# 🔹 A2. STORE inspection reports as objects, keyed by asset & date
# -----------------------------------------------------------
for r in insp.itertuples(index=False):
    key = f'inspections/{r.asset_id}/{r.inspection_date.date()}_{r.inspection_id}.json'
    store.put(key, {'inspector': r.inspector, 'result': r.result,
                    'health_score': r.health_score, 'cost_inr': int(r.cost_inr)})
print('objects stored:', len(store.meta))
# flat namespace, but prefixes act like folders:
a1_keys = store.list('inspections/A0001/')
print('objects under asset A0001:', len(a1_keys))
print('example key:', a1_keys[0])
print('fetch one object:', store.get(a1_keys[0]))

#### 🧪 EXERCISE 1 — Lifecycle tiering
Old reports are rarely read, so move them to a cheaper tier.
1. For every object whose date is before `2022-07-01` (parse it from the key), call `set_tier(key, 'archive')`.
2. Print how many objects are now `hot` vs `archive`, and the total bytes in each tier.
3. In a comment, explain what a real lifecycle policy automates and why retrieval from archive is slower/cheaper.

In [ ]:
# 1-2. archive old objects; report counts & bytes per tier
# YOUR CODE HERE

# 3. what a lifecycle policy automates: ...   (comment)

# Part B — Relational database (SQLite)
A relational store keeps data in typed tables linked by keys, with **ACID transactions** and fast **indexed** queries. SQLite gives us all of that in a single file.

In [ ]:
# -----------------------------------------------------------
# 🔹 B1. CREATE schema with a primary key and a foreign key
# -----------------------------------------------------------
DB = 'assets.db'
if os.path.exists(DB): os.remove(DB)
con = sqlite3.connect(DB)
con.execute('PRAGMA foreign_keys = ON')
con.execute('''CREATE TABLE assets (
    asset_id TEXT PRIMARY KEY, asset_type TEXT, site TEXT,
    install_date TEXT, criticality TEXT, manufacturer TEXT)''')
con.execute('''CREATE TABLE inspections (
    inspection_id TEXT PRIMARY KEY, asset_id TEXT, inspection_date TEXT,
    inspector TEXT, result TEXT, health_score REAL, duration_min INTEGER, cost_inr INTEGER,
    FOREIGN KEY (asset_id) REFERENCES assets(asset_id))''')
assets.astype(str).to_sql('assets', con, if_exists='append', index=False)
insp.astype({'inspection_date': str}).to_sql('inspections', con, if_exists='append', index=False)
con.commit()
print('loaded:', con.execute('SELECT COUNT(*) FROM assets').fetchone()[0], 'assets,',
      con.execute('SELECT COUNT(*) FROM inspections').fetchone()[0], 'inspections')

In [ ]:
# -----------------------------------------------------------
# 🔹 B2. ACID TRANSACTION — all-or-nothing writes
# Record an inspection AND bump the asset's manufacturer note together; if anything
# fails mid-way, the whole transaction rolls back, leaving the DB untouched.
# -----------------------------------------------------------
def add_inspection_txn(con, asset_id, inspection_id, score, force_fail=False):
    try:
        con.execute('BEGIN')
        con.execute('INSERT INTO inspections (inspection_id, asset_id, inspection_date, result, health_score)'
                    ' VALUES (?,?,?,?,?)', (inspection_id, asset_id, '2024-07-01', 'pass', score))
        if force_fail:
            raise ValueError('simulated failure after the first write')
        con.execute('UPDATE assets SET manufacturer = manufacturer WHERE asset_id = ?', (asset_id,))
        con.execute('COMMIT')
        return 'committed'
    except Exception as e:
        con.execute('ROLLBACK')
        return f'rolled back ({e})'

before = con.execute('SELECT COUNT(*) FROM inspections').fetchone()[0]
print('failing txn :', add_inspection_txn(con, 'A0001', 'I999001', 90, force_fail=True))
mid = con.execute('SELECT COUNT(*) FROM inspections').fetchone()[0]
print('good txn    :', add_inspection_txn(con, 'A0001', 'I999002', 90, force_fail=False))
after = con.execute('SELECT COUNT(*) FROM inspections').fetchone()[0]
print(f'counts: before={before}  after-failed-txn={mid}  after-good-txn={after}')
print('-> the failed transaction left NO partial row (atomicity).')

In [ ]:
# -----------------------------------------------------------
# 🔹 B3. INDEXING — make a filtered query fast
# Amplify the table so the timing difference is visible.
# -----------------------------------------------------------
big = pd.concat([insp.assign(inspection_id=insp.inspection_id + f'_{k}') for k in range(40)], ignore_index=True)
big.astype({'inspection_date': str}).to_sql('insp_big', con, if_exists='replace', index=False)
print('insp_big rows:', con.execute('SELECT COUNT(*) FROM insp_big').fetchone()[0])

def timed(sql, params=()):
    t = time.perf_counter(); con.execute(sql, params).fetchall(); return (time.perf_counter() - t) * 1000

q = 'SELECT * FROM insp_big WHERE asset_id = ?'
print('plan BEFORE index:', con.execute('EXPLAIN QUERY PLAN ' + q, ('A0123',)).fetchall()[0][-1])
t_before = np.mean([timed(q, ('A0123',)) for _ in range(5)])
con.execute('CREATE INDEX idx_assetid ON insp_big(asset_id)'); con.commit()
print('plan AFTER index :', con.execute('EXPLAIN QUERY PLAN ' + q, ('A0123',)).fetchall()[0][-1])
t_after = np.mean([timed(q, ('A0123',)) for _ in range(5)])
print(f'query time: {t_before:.2f} ms (scan)  ->  {t_after:.2f} ms (indexed)  =  {t_before/max(t_after,1e-6):.1f}x faster')

#### 🧪 EXERCISE 2 — Joins & a composite index
1. Write a SQL query that joins `assets` and `inspections` to list, per **site**, the average `health_score` of `high`-criticality assets. Run it with `pd.read_sql`.
2. Create a composite index on `insp_big(asset_id, inspection_date)` and use `EXPLAIN QUERY PLAN` to show a query filtering on both columns now SEARCHES using it.
3. In a comment, note the cost of indexes (slower writes, extra storage).

In [ ]:
# 1. join query: avg health by site for high-criticality assets
# 2. composite index + EXPLAIN QUERY PLAN
# YOUR CODE HERE

# 3. cost of indexes: ...   (comment)

# Part C — NoSQL document modelling
A document store (think MongoDB/DynamoDB) keeps related data **together** as one JSON document, optimised for fetch-by-key access rather than joins.

In [ ]:
# -----------------------------------------------------------
# 🔹 C1. The same data as ONE document per asset (denormalised)
# -----------------------------------------------------------
def build_asset_document(asset_id):
    a = assets[assets.asset_id == asset_id].iloc[0].to_dict()
    a['install_date'] = str(a['install_date'].date())
    rows = insp[insp.asset_id == asset_id]
    a['inspections'] = [{'date': str(r.inspection_date.date()), 'result': r.result,
                         'health_score': r.health_score} for r in rows.itertuples()]
    a['inspection_count'] = len(rows)
    return a

doc = build_asset_document('A0001')
print('document keys:', list(doc.keys()))
print('embedded inspections:', doc['inspection_count'])
print(json.dumps({k: doc[k] for k in ['asset_id', 'asset_type', 'site', 'inspection_count']}, indent=2))
print('first embedded inspection:', doc['inspections'][0])

In [ ]:
# -----------------------------------------------------------
# 🔹 C2. Access pattern: 'show everything about asset X'
# Document = ONE read. Relational = a join across two tables.
# -----------------------------------------------------------
docs = {aid: build_asset_document(aid) for aid in assets.asset_id}   # the 'document store'
d = docs['A0050']
print('document store: 1 key lookup ->', d['asset_id'], 'with', d['inspection_count'], 'inspections')
rel = pd.read_sql('SELECT a.*, i.inspection_date, i.result FROM assets a '
                  'JOIN inspections i ON a.asset_id=i.asset_id WHERE a.asset_id="A0050"', con)
print('relational: a join returning', len(rel), 'rows that the app must regroup')

#### 🧪 EXERCISE 3 — The trade-off
1. Compute the average number of embedded inspections per document (the 'fan-out').
2. Suppose an inspector's name changes. In a comment, explain why this one update is easy in the **relational** model (one row) but painful in the **document** model (rewrite every document that embeds that inspector) — the classic normalisation vs read-speed trade-off.

In [ ]:
# 1. average embedded inspections per document
# YOUR CODE HERE

# 2. update anomaly — relational vs document: ...   (comment)

# Part D — Sharding & replication
When one node can't hold the data, **shard** it: split rows across nodes by a shard key. We'll route assets across shards and watch what happens with a skewed key, then simulate a read **replica**.

In [ ]:
# -----------------------------------------------------------
# 🔹 D1. SHARD assets across nodes by a hash of the key
# -----------------------------------------------------------
import zlib
N_SHARDS = 4
def shard_of(key, n=N_SHARDS):
    return zlib.crc32(str(key).encode()) % n

shards = {i: [] for i in range(N_SHARDS)}
for aid in assets.asset_id:
    shards[shard_of(aid)].append(aid)
print('assets per shard (hash key -> even spread):', {i: len(v) for i, v in shards.items()})

In [ ]:
# -----------------------------------------------------------
# 🔹 D2. A BAD shard key causes a hot shard (skew)
# Sharding by 'site' (only 3 values, unevenly sized) overloads one node.
# -----------------------------------------------------------
site_shards = assets.groupby('site').size()
print('rows per shard when sharding by site:')
print(site_shards.to_string())
print('-> Plant_A is a HOT shard; the cluster is only as fast as its busiest node.')

In [ ]:
# -----------------------------------------------------------
# 🔹 D3. REPLICATION & eventual consistency (a tiny simulation)
# Primary takes writes; a replica lags slightly before catching up.
# -----------------------------------------------------------
primary = {'A0001_status': 'ok'}
replica = dict(primary)        # in sync to start
primary['A0001_status'] = 'fault'      # a write hits the primary
print('immediately after write:')
print('  read from primary:', primary['A0001_status'])
print('  read from replica:', replica['A0001_status'], '(stale -> eventual consistency)')
replica.update(primary)        # replication catches up
print('after replication catches up, replica:', replica['A0001_status'])

#### 🧪 EXERCISE 4 — Range sharding & rebalancing
1. Shard the assets by the **first character range** of `asset_id` instead of by hash (e.g. split the sorted id space into `N_SHARDS` contiguous buckets). Print the per-shard counts.
2. In a comment, compare hash vs range sharding: range supports efficient range scans but risks hotspots on sequential keys; hash spreads evenly but scatters ranges. Which would you pick for time-series inspection data, and why?

In [ ]:
# 1. range-shard the asset_id space; show per-shard counts
# YOUR CODE HERE

# 2. hash vs range sharding choice: ...   (comment)

In [ ]:
con.close()
print('database connection closed.')

#📘 Summary

| Topic | What you built |
| ----- | -------------- |
| Object storage | a keyed bucket with metadata and storage tiers |
| Relational + ACID | typed tables, a transaction that rolls back atomically |
| Indexing | measured a real scan→index speedup and read the query plan |
| Document model | one JSON doc per asset; one-read access vs joins |
| Sharding | hash spread vs a skewed hot shard; range sharding |
| Replication | eventual consistency from replica lag |

**Core lesson:** there's no universal 'best' store. Match the **storage type** (object/block/file) and **database model** (relational vs document) to how the data is written and read, index the columns you query, and — once you scale out — choose a shard key that spreads load and accept the consistency trade-offs replication brings.

**Next — U24 (Part 2):** scaling, caching, query optimisation and security.